In [4]:
import cv2
from ultralytics import YOLO
from collections import defaultdict

model = YOLO('yolo11l.pt')
class_list = model.names

cap = cv2.VideoCapture('test_videos/sample.mp4')
line_y_red = 430

class_counts = defaultdict(int)
crossed_ids = set()

# Video writer setup
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, fps, (frame_width, frame_height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True)

    if results[0].boxes.data is not None:
        boxes = results[0].boxes.xyxy.cpu()
        track_ids = results[0].boxes.id.int().cpu().tolist()
        class_indices = results[0].boxes.cls.int().cpu().tolist()
        confidences = results[0].boxes.conf.cpu()
        cv2.line(frame, (690, line_y_red), (1130, line_y_red), (0, 0, 255), 3)

        for box, track_id, class_idx, conf in zip(boxes, track_ids, class_indices, confidences):
            x1, y1, x2, y2 = map(int, box)
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            class_name = class_list[class_idx]
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
            cv2.putText(frame, f"ID:{track_id} {class_name}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            if cy > line_y_red and track_id not in crossed_ids:
                crossed_ids.add(track_id)
                class_counts[class_name] += 1

    y_offset = 30
    for class_name, count in class_counts.items():
        cv2.putText(frame, f"{class_name}: {count}", (0, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        y_offset += 30

    out.write(frame)  # Save frame to output video

    cv2.imshow("YOLO Object Tracking and Counting", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()  # Release the video writer
cv2.destroyAllWindows()


0: 384x640 6 persons, 10 cars, 3 motorcycles, 3 buss, 1 truck, 343.8ms
Speed: 2.1ms preprocess, 343.8ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 persons, 10 cars, 3 motorcycles, 3 buss, 1 truck, 403.3ms
Speed: 4.2ms preprocess, 403.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 persons, 10 cars, 3 motorcycles, 3 buss, 1 truck, 421.0ms
Speed: 4.0ms preprocess, 421.0ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 persons, 10 cars, 3 motorcycles, 3 buss, 1 truck, 357.9ms
Speed: 2.4ms preprocess, 357.9ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 persons, 10 cars, 4 motorcycles, 3 buss, 1 truck, 344.5ms
Speed: 2.5ms preprocess, 344.5ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 persons, 10 cars, 4 motorcycles, 3 buss, 1 truck, 355.2ms
Speed: 2.4ms preprocess, 355.2ms inference, 1.1ms postprocess per image at sha